In [6]:
using Pkg
Pkg.status()


Project LMP v0.1.0
Status `C:\Users\_s2220186\Desktop\Julia\LMP\Project.toml`
  [052768ef] CUDA v5.9.6
  [0b91fe84] DisplayAs v0.1.6
  [a461e10c] Gaugefields v0.6.16
  [7073ff75] IJulia v1.34.0
⌅ [033835bb] JLD2 v0.4.55
  [019239df] LatticeDiracOperators v0.5.11
⌃ [91a5bcdd] Plots v1.40.7
  [d330b81b] PyPlot v2.11.6
  [295af30f] Revise v3.13.2
  [90137ffa] StaticArrays v1.9.16
  [f3c8d4fe] Wilsonloop v0.1.5
  [37e2e46d] LinearAlgebra v1.11.0
  [9a3f8284] Random v1.11.0
Info Packages marked with ⌃ and ⌅ have new versions available. Those with ⌃ may be upgradable, but those with ⌅ are restricted by compatibility constraints from upgrading. To see why use `status --outdated`


In [2]:
using CUDA
using LinearAlgebra
using Random
using LatticeDiracOperators
using InteractiveUtils
using Gaugefields
import Gaugefields.AbstractGaugefields_module:getvalue,setvalue!

println("Test for 1/D for Wilson CUDA")

versioninfo()

version = Base.pkgversion(Gaugefields)
println("version of Gaugefields : ", version)
version = Base.pkgversion(LatticeDiracOperators)
println("version of LatticeDiracOperators: ", version)

const pes =  (4, 4, 4, 4)
const cuda = true
const numtrj_in = 2 # the number of operations to take average of performance
const MDsteps_in = 0 # dummy

const βin = 6.0
const NXin = 4
const NYin = 4
const NZin = 4
const NTin = 8
const κin = 0.141139

header = """
Parameters
const blocks = $pes
const cuda = $cuda
const numtrj_in = $numtrj_in
const MDsteps_in = $MDsteps_in

const βin = $βin
const NXin = $NXin
const NYin = $NYin
const NZin = $NZin
const NTin = $NTin
const κin = $κin
"""

function test1()
    NX = NXin
    NY = NYin
    NZ = NZin
    NT = NTin
    Nwing = 0
    Dim = 4
    NC = 3


        PEs = pes#(1,1,1,2)
        U = Initialize_Gaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "cold",cuda=true,blocks=PEs)


    Ucpu = Initialize_Gaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "cold")
    substitute_U!(Ucpu,U)

    Random.seed!(123+get_myrank(U[1]))

    if get_myrank(U) == 0
        println(header) # print header
    end

    snet = GaugeAction(U)
    plaqloop = make_loops_fromname("plaquette")
    append!(plaqloop,plaqloop')
    β = βin/2
    push!(snet,β,plaqloop)

    #show(snet)

    Random.seed!(123)
    x = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
    xcpu = Initialize_pseudofermion_fields(Ucpu[1],"Wilson",nowing=true)

    params = Dict()
    params["Dirac_operator"] = "Wilson"
    params["κ"] = κin
    params["faster version"] = true
    params["eps_CG"] = 1.0e-16
    params["verbose_level"] = 3 # to show CG counts

    D = Dirac_operator(U,x,params)

    setvalue_fermion!(xcpu,10.0,1,1,1,1,1,1)
    setvalue_fermion!(xcpu,im,1,2,1,1,1,1)
    setvalue_fermion!(xcpu,2im,1,3,1,1,1,1)
    setvalue_fermion!(xcpu,3im,1,4,1,1,1,1)
    setvalue_fermion!(xcpu,10.0,1,1,2,1,1,1)
    setvalue_fermion!(xcpu,im,1,2,1,1,4,1)
    setvalue_fermion!(xcpu,2im,1,3,1,4,1,1)
    setvalue_fermion!(xcpu,3im,1,4,1,1,1,1)

    substitute_fermion!(x,xcpu)


    b = similar(x)
    y = similar(x)
    @time solve_DinvX!(b,D,x)
    @time mul!(b,D,x)
    @time mul!(b,D,x)

    println_verbose_level1(U[1],"Benchmark!!")
    @time solve_DinvX!(y,D,b)


    println_verbose_level1(U[1],"We check whether y = D^-1 (Dx) = x")
    ycpu = similar(xcpu)
    substitute_fermion!(ycpu,y)

    println(ycpu[1,1,1,1,1,1])
    println(ycpu[1,2,1,1,1,1])

    println_verbose_level1(U[1],"several times!!!")
    elptime = 0.0
    tmp = 0.0
    for itrj=1:numtrj_in
        println_verbose_level1(U[1],"itrj = $itrj")
        gauss_distribution_fermion!(x)
        tmp = @elapsed solve_DinvX!(y,D,x)
        println("# CG solved $tmp sec")
        elptime+=tmp
    end
    elptime/=numtrj_in
    println_verbose_level1(U[1],"$elptime #Dtime")

    return
end



[ Info: Threads backend loaded with 1 threads


Test for 1/D for Wilson CUDA
Julia Version 1.11.7
Commit f2b3dbda30 (2025-09-08 12:10 UTC)
Build Info:
  Official https://julialang.org/ release
Platform Info:
  OS: Windows (x86_64-w64-mingw32)
  CPU: 32 × AMD Ryzen 9 7950X3D 16-Core Processor          
  WORD_SIZE: 64
  LLVM: libLLVM-16.0.6 (ORCJIT, znver4)
Threads: 1 default, 0 interactive, 1 GC (on 32 virtual cores)
version of Gaugefields : 0.6.16
version of LatticeDiracOperators: 0.5.11


test1 (generic function with 1 method)

In [3]:
CUDA.allowscalar(true)
test1()

┌ Warning: It's not recommended to use allowscalar([true]) to allow scalar indexing.
│ Instead, use `allowscalar() do end` or `@allowscalar` to denote exactly which operations can use scalar operations.
└ @ GPUArraysCore C:\Users\_s2220186\.julia\packages\GPUArraysCore\aNaXo\src\GPUArraysCore.jl:184


Parameters
const blocks = (4, 4, 4, 4)
const cuda = true
const numtrj_in = 2
const MDsteps_in = 0

const βin = 6.0
const NXin = 4
const NYin = 4
const NZin = 4
const NTin = 8
const κin = 0.141139

--------------------------------------
bicg method
1-th eps: 67.29568847233656
2-th eps: 31.25785449779739
3-th eps: 17.541435275724773
4-th eps: 11.109145205874176
5-th eps: 7.4882259327048075
6-th eps: 4.954993817869864
7-th eps: 3.4172616314843296
8-th eps: 2.3865549942475326
9-th eps: 1.7788530855070785
10-th eps: 1.3393974906244415
11-th eps: 1.0462248467091433
12-th eps: 0.8412682139136385
13-th eps: 0.6799721688850414
14-th eps: 0.5510415455349469
15-th eps: 0.447547979316807
16-th eps: 0.34279091663928285
17-th eps: 0.20713603201464476
18-th eps: 0.12493718072760934
19-th eps: 0.08694585083387493
20-th eps: 0.06363628952343309
21-th eps: 0.0380092847428426
22-th eps: 0.021934060253211346
23-th eps: 0.010990827113378928
24-th eps: 0.004979911893140306
25-th eps: 0.0026759356785575937
2

In [4]:
using Gaugefields
using LatticeDiracOperators
using LinearAlgebra
using InteractiveUtils
using Random
using CUDA

const pes =  (4, 4, 4, 4)
const cuda = true

function MDtest!(gauge_action,U,Dim,fermi_action,η,ξ)
    p = initialize_TA_Gaugefields(U) #This is a traceless-antihermitian gauge fields. This has NC^2-1 real coefficients.
    Uold = similar(U)
    substitute_U!(Uold,U)
    MDsteps = 10
    temp1 = similar(U[1])
    temp2 = similar(U[1])
    comb = 6
    factor = 1/(comb*U[1].NV*U[1].NC)
    numaccepted = 0
    Random.seed!(123)

    numtrj = 10
    for itrj = 1:numtrj
        @time accepted = MDstep!(gauge_action,U,p,MDsteps,Dim,Uold,fermi_action,η,ξ)
        numaccepted += ifelse(accepted,1,0)

        plaq_t = calculate_Plaquette(U,temp1,temp2)*factor
        println("$itrj plaq_t = $plaq_t")
        println("acceptance ratio ",numaccepted/itrj)
    end
end

function calc_action(gauge_action,U,p)
    NC = U[1].NC
    Sg = -evaluate_GaugeAction(gauge_action,U)/NC #evaluate_GaugeAction(gauge_action,U) = tr(evaluate_GaugeAction_untraced(gauge_action,U))
    Sp = p*p/2
    S = Sp + Sg
    return real(S)
end


function MDstep!(gauge_action,U,p,MDsteps,Dim,Uold,fermi_action,η,ξ)
    Δτ = 1/MDsteps
    NC,_,NN... = size(U[1])

    gauss_distribution!(p)

    substitute_U!(Uold,U)
    gauss_sampling_in_action!(ξ,U,fermi_action)
    sample_pseudofermions!(η,U,fermi_action,ξ)
    Sfold = real(dot(ξ,ξ))
    println("Sfold = $Sfold")

    Sold = calc_action(gauge_action,U,p) + Sfold
    println("Sold = ",Sold)

    for itrj=1:MDsteps
        U_update!(U,p,0.5,Δτ,Dim,gauge_action)

        P_update!(U,p,1.0,Δτ,Dim,gauge_action)
        P_update_fermion!(U,p,1.0,Δτ,Dim,gauge_action,fermi_action,η)

        U_update!(U,p,0.5,Δτ,Dim,gauge_action)
    end
    Sfnew = evaluate_FermiAction(fermi_action,U,η)
    println("Sfnew = $Sfnew")
    Snew = calc_action(gauge_action,U,p) + Sfnew

    println("Sold = $Sold, Snew = $Snew")
    println("Snew - Sold = $(Snew-Sold)")

    accept = exp(Sold - Snew) >= rand()

    #ratio = min(1,exp(Snew-Sold))
    if accept != true #rand() > ratio
        substitute_U!(U,Uold)
        return false
    else
        return true
    end
end

function U_update!(U,p,ϵ,Δτ,Dim,gauge_action)
    temps = get_temporary_gaugefields(gauge_action)
    temp1 = temps[1]
    temp2 = temps[2]
    expU = temps[3]
    W = temps[4]

    for μ=1:Dim
        exptU!(expU,ϵ*Δτ,p[μ],[temp1,temp2])
        mul!(W,expU,U[μ])
        substitute_U!(U[μ],W)

    end
end

function P_update!(U,p,ϵ,Δτ,Dim,gauge_action) # p -> p +factor*U*dSdUμ
    NC = U[1].NC
    temps = get_temporary_gaugefields(gauge_action)
    dSdUμ = temps[end]
    factor =  -ϵ*Δτ/(NC)

    for μ=1:Dim
        calc_dSdUμ!(dSdUμ,gauge_action,μ,U)
        mul!(temps[1],U[μ],dSdUμ) # U*dSdUμ
        Traceless_antihermitian_add!(p[μ],factor,temps[1])
    end
end

function P_update_fermion!(U,p,ϵ,Δτ,Dim,gauge_action,fermi_action,η)  # p -> p +factor*U*dSdUμ
    #NC = U[1].NC
    temps = get_temporary_gaugefields(gauge_action)
    UdSfdUμ = temps[1:Dim]
    factor =  -ϵ*Δτ

    calc_UdSfdU!(UdSfdUμ,fermi_action,U,η)

    for μ=1:Dim
        Traceless_antihermitian_add!(p[μ],factor,UdSfdUμ[μ])
        #println(" p[μ] = ", p[μ][1,1,1,1,1])
    end
end

function test1()
    NX = 4
    NY = 4
    NZ = 4
    NT = 4
    Nwing = 0
    Dim = 4
    NC = 3

    #U = Initialize_4DGaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "cold")

    if cuda
        PEs = pes#(1,1,1,2)
        U = Initialize_Gaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "cold",cuda=true,blocks=PEs)
    else
        U = Initialize_Gaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "cold")
    end
    Ucpu = Initialize_Gaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "cold")
    substitute_U!(Ucpu,U)

    Random.seed!(123+get_myrank(U[1]))

    if get_myrank(U) == 0
        # println(header) # print header
    end

    gauge_action = GaugeAction(U)
    plaqloop = make_loops_fromname("plaquette")
    append!(plaqloop,plaqloop')
    β = 5.5/2
    push!(gauge_action,β,plaqloop)

    show(gauge_action)

    #x = Initialize_pseudofermion_fields(U[1],"Wilson")
#--
    #Random.seed!(123)
    x = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
    xcpu = Initialize_pseudofermion_fields(Ucpu[1],"Wilson",nowing=true)

    params = Dict()
    params["Dirac_operator"] = "Wilson"
    params["κ"] = κin
    params["faster version"] = true
    params["eps_CG"] = 1.0e-16
    params["verbose_level"] = 2 # set 3 to show CG counts

    D = Dirac_operator(U,x,params)

    setvalue_fermion!(xcpu,10.0,1,1,1,1,1,1)
    setvalue_fermion!(xcpu,im,1,2,1,1,1,1)
    setvalue_fermion!(xcpu,2im,1,3,1,1,1,1)
    setvalue_fermion!(xcpu,3im,1,4,1,1,1,1)
    setvalue_fermion!(xcpu,10.0,1,1,2,1,1,1)
    setvalue_fermion!(xcpu,im,1,2,1,1,4,1)
    setvalue_fermion!(xcpu,2im,1,3,1,4,1,1)
    setvalue_fermion!(xcpu,3im,1,4,1,1,1,1)

    substitute_fermion!(x,xcpu)
#--
    parameters_action = Dict()
    fermi_action = FermiAction(D,parameters_action)

    y = similar(x)

    MDtest!(gauge_action,U,Dim,fermi_action,x,y)

end


test1()

----------------------------------------------
Structure of the actions for Gaugefields
num. of terms: 1
-------------------------------
      1-st term: 
          coefficient: 2.75 + 0.0im
      -------------------------
1-st loop
L"$U_{1}(n)U_{2}(n+e_{1})U^{\dagger}_{1}(n+e_{2})U^{\dagger}_{2}(n)$"	
2-nd loop
L"$U_{1}(n)U_{3}(n+e_{1})U^{\dagger}_{1}(n+e_{3})U^{\dagger}_{3}(n)$"	
3-rd loop
L"$U_{1}(n)U_{4}(n+e_{1})U^{\dagger}_{1}(n+e_{4})U^{\dagger}_{4}(n)$"	
4-th loop
L"$U_{2}(n)U_{3}(n+e_{2})U^{\dagger}_{2}(n+e_{3})U^{\dagger}_{3}(n)$"	
5-th loop
L"$U_{2}(n)U_{4}(n+e_{2})U^{\dagger}_{2}(n+e_{4})U^{\dagger}_{4}(n)$"	
6-th loop
L"$U_{3}(n)U_{4}(n+e_{3})U^{\dagger}_{3}(n+e_{4})U^{\dagger}_{4}(n)$"	
7-th loop
L"$U_{2}(n)U_{1}(n+e_{2})U^{\dagger}_{2}(n+e_{1})U^{\dagger}_{1}(n)$"	
8-th loop
L"$U_{3}(n)U_{1}(n+e_{3})U^{\dagger}_{3}(n+e_{1})U^{\dagger}_{1}(n)$"	
9-th loop
L"$U_{4}(n)U_{1}(n+e_{4})U^{\dagger}_{4}(n+e_{1})U^{\dagger}_{1}(n)$"	
10-th loop
L"$U_{3}(n)U_{2}(n+e_{3})U^{\dagger}_

In [5]:
a=5
b=similar(a)
println("a = $a, b = $b")

LoadError: MethodError: no method matching similar(::Int64)
The function `similar` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  similar([91m::SentinelArrays.MissingVector[39m, [91m::Type{Missing}[39m, [91m::NTuple{N, Int64}[39m) where N
[0m[90m   @[39m [35mSentinelArrays[39m [90mC:\Users\_s2220186\.julia\packages\SentinelArrays\bupWd\src\[39m[90m[4mmissingvector.jl:20[24m[39m
[0m  similar([91m::ZMQ.Message[39m, [91m::Type{T}[39m, [91m::NTuple{N, Int64} where N[39m) where T
[0m[90m   @[39m [36mZMQ[39m [90mC:\Users\_s2220186\.julia\packages\ZMQ\yNY0H\src\[39m[90m[4mmessage.jl:162[24m[39m
[0m  similar([91m::Type{SA}[39m) where SA<:StaticArraysCore.StaticArray
[0m[90m   @[39m [32mStaticArrays[39m [90mC:\Users\_s2220186\.julia\packages\StaticArrays\IZgIP\src\[39m[90m[4mabstractarray.jl:117[24m[39m
[0m  ...
